## Global Airports Dataset - EDA 

Exploratory analysis of  ..... The airports data retrived from [Kaggle](https://www.kaggle.com/datasets/samvelkoch/global-airports-iata-icao-timezone-geo?resource=download&select=airports.csv), adding the **TimeZone** Information into the final dataset. 

In [20]:
import numpy as np 
import pandas as pd 
data = pd.read_csv("airports.csv")

## Overview

In [21]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6393 entries, 0 to 6392
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   AirportName         6393 non-null   str    
 1   IATA                6373 non-null   str    
 2   ICAO                5091 non-null   str    
 3   TimeZone            6305 non-null   str    
 4   City_Name           6393 non-null   str    
 5   City_IATA           6331 non-null   str    
 6   UTC_Offset_Hours    6305 non-null   float64
 7   UTC_Offset_Seconds  6305 non-null   float64
 8   Country_CodeA2      6379 non-null   str    
 9   Country_CodeA3      6393 non-null   str    
 10  Country_Name        6393 non-null   str    
 11  GeoPointLat         6393 non-null   float64
 12  GeoPointLong        6393 non-null   float64
dtypes: float64(4), str(9)
memory usage: 649.4 KB


In [22]:
data.head()

,AirportName,IATA,ICAO,TimeZone,City_Name,City_IATA,UTC_Offset_Hours,UTC_Offset_Seconds,Country_CodeA2,Country_CodeA3,Country_Name,GeoPointLat,GeoPointLong
0,Pilanesberg Intl,NTY,FAPN,Africa/Johannesburg,Sun City,NTY,2.0,7200.0,ZA,ZAF,South Africa,-25.333822,27.173358
1,Clovis Muni,CVN,KCVN,America/Denver,"Clovis, New Mexico",CVN,-6.0,-21600.0,US,USA,United States of America,34.425139,-103.079278
2,Cannon Afb,CVS,KCVS,America/Denver,"Clovis, New Mexico",CVN,-6.0,-21600.0,US,USA,United States of America,34.382775,-103.322147
3,Scammon Bay Airport,SCM,PACM,America/Nome,"Scammon Bay, Alaska",SCM,-8.0,-28800.0,US,USA,United States of America,61.845278,-165.571389
4,Kapit,KPI,NaN,Asia/Kuching,Kapit,KPI,8.0,28800.0,MY,MYS,Malaysia,2.017000,112.950000


## Missingness

In [23]:
total_entries = data.shape[0]

In [24]:
(data.isna().sum() *100 / total_entries).round(2).rename("missing_%")

AirportName            0.00
IATA                   0.31
ICAO                  20.37
TimeZone               1.38
City_Name              0.00
City_IATA              0.97
UTC_Offset_Hours       1.38
UTC_Offset_Seconds     1.38
Country_CodeA2         0.22
Country_CodeA3         0.00
Country_Name           0.00
GeoPointLat            0.00
GeoPointLong           0.00
Name: missing_%, dtype: float64

## Pre-processing

- remove the AirportName empty left/right part of the string
- select those with identical ICAO

In [101]:
# Remove the blanks part of Airport Name String
data["AirportName"] = data["AirportName"].str.lstrip()
data["AirportName"] = data["AirportName"].str.rstrip()

In [110]:
data_na_icao = data.loc[:, ["ICAO"]].isnull() == False

print("Duplicated Airport Name:", data.loc[:,["ICAO"]][data_na_icao].duplicated().sum())

Duplicated Airport Name: 1303


In [111]:
data_na_icao = data[data["ICAO"].notna()]
icao_dup = data_na_icao["ICAO"].duplicated(keep=False)

print("Duplicated ICAO values:", data_na_icao.loc[icao_dup, "ICAO"].nunique())
print("Rows involved:", icao_dup.sum())
data_na_icao.loc[icao_dup, ["AirportName", "IATA", "ICAO", "City_Name", "Country_Name"]].sort_values("ICAO")

Duplicated ICAO values: 2
Rows involved: 4


,AirportName,IATA,ICAO,City_Name,Country_Name
5766,Schonefeld,SXF,EDDB,Berlin,Germany
5767,Berlin Brandenburg Willy Brandt,BER,EDDB,Berlin,Germany
4096,Dortmund,DTM,EDLW,Dortmund,Germany
4097,HBF RAILWAY STATION,DTZ,EDLW,Dortmund,Germany


In [ ]:
# Remove duplicated ICAO entries: old Berlin Schonefeld (merged into BER) and Dortmund railway station
drop_mask = ((data["ICAO"] == "EDDB") & (data["AirportName"] == "Schonefeld")) | \
            ((data["ICAO"] == "EDLW") & (data["AirportName"] == "HBF RAILWAY STATION"))

print("Rows removed:", drop_mask.sum())
data = data[~drop_mask].reset_index(drop=True)
print("Remaining rows:", data.shape[0])

In [99]:
# delete AirportName, CountryName, ICAO, City_Name  duplicated data 
#

duplicated_data = data.loc[:,["AirportName", "Country_Name", "ICAO", "City_Name", "GeoPointLat" , "GeoPointLong"]][data.loc[:,["AirportName", "GeoPointLat" , "GeoPointLong"]].duplicated(keep = False)]

airport_names_du = duplicated_data.AirportName.unique()


# name = airport_names_du[20] # 20
# duplicated_data.query(f"AirportName == @name")

duplicated_data

,AirportName,Country_Name,ICAO,City_Name,GeoPointLat,GeoPointLong
476,Long island,Australia,NaN,Long Island,0.0,0.0
616,BUS STATION,Austria,NaN,Woergl,0.0,0.0
666,RAILWAY STATION,Spain,NaN,Murcia,0.0,0.0
830,HBF RAILWAY STATION,Germany,NaN,Ulm,0.0,0.0
1754,CENTRAL RAILWAY STN,Italy,NaN,Bologna,0.0,0.0
1879,HBF RAILWAY STATION,Germany,NaN,Mannheim,0.0,0.0
1956,RAILWAY STATION,Austria,NaN,Innsbruck,0.0,0.0
2088,VALENCIA RAILWAY STN,Spain,NaN,Valencia,0.0,0.0
2193,BUS STATION,Italy,NaN,Reggio Nell Emilia,0.0,0.0
2820,RAILWAY STATION,Italy,NaN,Trieste,0.0,0.0


In [ ]:
# data fusion 
# 經緯度將 (0,0) 換成 NAN 讓他不參與 data. resolution 投票
# 檢查經緯度是否在 +- 180, +-90



48

In [ ]:
data.loc[:,""]